In [1]:
from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

from scipy import sparse


from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score

import joblib
RANDOM_STATE = 42

## Load Week 6 Model Data

The final Week 6 feature matrices are loaded directly instead of
re-running preprocessing and spatial feature engineering.

Using the saved matrices ensures that XGBoost and the previous models
are evaluated on exactly the same rows and features.

In [6]:
# Path to the final model-ready data saved during Week 6
data_dir = Path(
    "../data/week6_final_model_data"
)

x_train_path = data_dir / "X_train_week6.npz"
x_test_path = data_dir / "X_test_week6.npz"
y_train_path = data_dir / "y_train_week6.npy"
y_test_path = data_dir / "y_test_week6.npy"


# Confirm that all required files exist
required_paths = [
    x_train_path,
    x_test_path,
    y_train_path,
    y_test_path
]

missing_files = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following Week 6 files were not found:\n"
        + "\n".join(missing_files)
    )

print("All Week 6 data files were found.")
print("Data folder:", data_dir.resolve())

All Week 6 data files were found.
Data folder: D:\IDX_Exchange\IDX-Exchange-summer-2026\data\week6_final_model_data


In [7]:
# Load sparse feature matrices
X_train = sparse.load_npz(
    x_train_path
).tocsr().astype(
    np.float32,
    copy=False
)

X_test = sparse.load_npz(
    x_test_path
).tocsr().astype(
    np.float32,
    copy=False
)


# Load target arrays
y_train = np.load(
    y_train_path
).astype(
    np.float32,
    copy=False
)

y_test = np.load(
    y_test_path
).astype(
    np.float32,
    copy=False
)


print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (129813, 1165)
X_test shape: (12024, 1165)
y_train shape: (129813,)
y_test shape: (12024,)


## Data Validation

The following checks confirm that:

- The number of feature rows matches the number of targets.
- Training and testing matrices contain the same number of columns.
- The matrices contain valid finite values.
- The Week 7 models use the expected Week 6 dataset.

In [8]:
# Check row alignment
assert X_train.shape[0] == y_train.shape[0], (
    "Training features and target are not aligned."
)

assert X_test.shape[0] == y_test.shape[0], (
    "Testing features and target are not aligned."
)


# Check feature consistency
assert X_train.shape[1] == X_test.shape[1], (
    "Training and testing matrices have different feature counts."
)


# Confirm expected Week 6 dimensions
assert X_train.shape == (129813, 1165), (
    f"Unexpected training shape: {X_train.shape}"
)

assert X_test.shape == (12024, 1165), (
    f"Unexpected testing shape: {X_test.shape}"
)


# Check target values
assert np.isfinite(y_train).all(), (
    "y_train contains NaN or infinite values."
)

assert np.isfinite(y_test).all(), (
    "y_test contains NaN or infinite values."
)


# Check sparse matrix stored values
assert np.isfinite(X_train.data).all(), (
    "X_train contains NaN or infinite values."
)

assert np.isfinite(X_test.data).all(), (
    "X_test contains NaN or infinite values."
)


print("Data validation completed successfully.")
print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

Data validation completed successfully.
Training observations: 129813
Testing observations: 12024
Number of features: 1165


## Baseline XGBoost

A baseline XGBoost model is trained before hyperparameter tuning.

This provides a reference point for determining whether tuning improves
model performance.

In [9]:
# Build the baseline XGBoost regression model
baseline_xgb = XGBRegressor(
    objective="reg:squarederror",

    # Number of sequential trees
    n_estimators=100,

    # Maximum depth of each tree
    max_depth=6,

    # Contribution of each new tree
    learning_rate=0.1,

    # Faster and more memory-efficient tree construction
    tree_method="hist",

    # Evaluation loss used internally
    eval_metric="rmse",

    # Reproducibility
    random_state=RANDOM_STATE,

    # Limit CPU usage to reduce memory pressure
    n_jobs=1
)

baseline_xgb

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [10]:
# Measure training time
baseline_start_time = time.time()


# Train baseline model
baseline_xgb.fit(
    X_train,
    y_train
)


baseline_training_time = (
    time.time()
    -
    baseline_start_time
)


print(
    f"Baseline XGBoost training completed in "
    f"{baseline_training_time / 60:.2f} minutes."
)

Baseline XGBoost training completed in 0.09 minutes.


In [11]:
# Generate predictions
baseline_train_predictions = baseline_xgb.predict(
    X_train
)

baseline_test_predictions = baseline_xgb.predict(
    X_test
)


# Calculate R²
baseline_train_r2 = r2_score(
    y_train,
    baseline_train_predictions
)

baseline_test_r2 = r2_score(
    y_test,
    baseline_test_predictions
)

baseline_gap = (
    baseline_train_r2
    -
    baseline_test_r2
)


print(
    f"Baseline XGBoost training R²: "
    f"{baseline_train_r2:.4f}"
)

print(
    f"Baseline XGBoost testing R²: "
    f"{baseline_test_r2:.4f}"
)

print(
    f"Baseline train-test gap: "
    f"{baseline_gap:.4f}"
)

Baseline XGBoost training R²: 0.8897
Baseline XGBoost testing R²: 0.5192
Baseline train-test gap: 0.3705


## Time-Based Cross-Validation

The training rows were previously arranged chronologically.

TimeSeriesSplit is used so that earlier observations are used to predict
later observations within each validation fold.

This is more appropriate than random cross-validation because the final
business objective is to use historical sales to predict future sales.

In [12]:
# Three time-ordered validation folds
time_cv = TimeSeriesSplit(
    n_splits=3
)

print("Number of CV splits:", time_cv.get_n_splits())

Number of CV splits: 3


In [13]:
# Display the size of each time-based fold
cv_fold_summary = []

for fold_number, (train_position, validation_position) in enumerate(
    time_cv.split(X_train),
    start=1
):
    cv_fold_summary.append(
        {
            "Fold": fold_number,
            "Training Rows": len(train_position),
            "Validation Rows": len(validation_position),
            "First Training Position": train_position[0],
            "Last Training Position": train_position[-1],
            "First Validation Position": validation_position[0],
            "Last Validation Position": validation_position[-1]
        }
    )

cv_fold_summary_df = pd.DataFrame(
    cv_fold_summary
)

display(cv_fold_summary_df)

,Fold,Training Rows,Validation Rows,First Training Position,Last Training Position,First Validation Position,Last Validation Position
0,1,32454,32453,0,32453,32454,64906
1,2,64907,32453,0,64906,64907,97359
2,3,97360,32453,0,97359,97360,129812


## Hyperparameter Tuning

The Week 7 task requires light tuning of:

- `max_depth`
- `learning_rate`
- `n_estimators`

A small parameter grid is used because the dataset contains nearly
130,000 training observations and 1,165 features.

In [18]:
# Base XGBoost estimator used by GridSearchCV
xgb_tuning_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=1
)


# Light parameter grid
xgb_param_grid = {
    "max_depth": [
        6,
        7,
        8
    ],

    "learning_rate": [
         0.1,
         0.15,
         0.2
    ],

    "n_estimators": [
        200,
        300,
        400
    ]
}


# Number of parameter combinations
number_of_combinations = (
    len(xgb_param_grid["max_depth"])
    *
    len(xgb_param_grid["learning_rate"])
    *
    len(xgb_param_grid["n_estimators"])
)

number_of_fits = (
    number_of_combinations
    *
    time_cv.get_n_splits()
)


print("Parameter combinations:", number_of_combinations)
print("CV folds:", time_cv.get_n_splits())
print("Estimated total fits:", number_of_fits)

Parameter combinations: 27
CV folds: 3
Estimated total fits: 81


In [19]:
# Build the grid search
xgb_grid_search = GridSearchCV(
    estimator=xgb_tuning_model,
    param_grid=xgb_param_grid,

    # Optimize R²
    scoring="r2",

    # Preserve chronological order
    cv=time_cv,

    # One parallel job to reduce memory use
    n_jobs=1,

    # Avoid creating too many jobs in advance
    pre_dispatch=1,

    # Show progress
    verbose=2,

    # Stop immediately if a fit fails
    error_score="raise",

    # Save training scores for overfitting comparison
    return_train_score=True,

    # Automatically refit the best model on all training data
    refit=True
)

xgb_grid_search

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.15, ...], 'max_depth': [6, 7, ...], 'n_estimators': [200, 300, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"pre_dispatch pre_dispatch: int, or str, default='2*n_jobs'Controls the number of jobs that get dispatched during parallelexecution. Reducing this number can be useful to avoid anexplosion of memory consumption when more jobs get dispatchedthan CPUs can process. This parameter can be:- None, in which case all the jobs are immediately created and spawned. Use this for lightweight and fast-running jobs, to avoid delays due to on-demand spawning of the jobs- An int, giving the exact number of total jobs that are spawned- A str, giving an expression as a function of n_jobs, as in '2*n_jobs'",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get in

In [20]:
# Run light hyperparameter tuning
tuning_start_time = time.time()


xgb_grid_search.fit(
    X_train,
    y_train
)


tuning_training_time = (
    time.time()
    -
    tuning_start_time
)


print(
    f"Hyperparameter tuning completed in "
    f"{tuning_training_time / 60:.2f} minutes."
)

Fitting 3 folds for each of 27 candidates, totalling 81 fits
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=200; total time=   2.3s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=200; total time=   3.0s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=200; total time=   3.9s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=300; total time=   3.2s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=300; total time=   4.4s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=300; total time=   5.9s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=400; total time=   3.7s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=400; total time=   5.5s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=400; total time=   7.4s
[CV] END ...learning_rate=0.1, max_depth=7, n_estimators=200; total time=   3.1s
[CV] END ...learning_rate=0.1, max_depth=7, n_estimators=200; total time=   4.6s
[CV] END ...learning_rate=0.1, max_depth=7, n_es

In [21]:
print("Best XGBoost parameters:")
print(xgb_grid_search.best_params_)

print(
    f"Best mean cross-validation R²: "
    f"{xgb_grid_search.best_score_:.4f}"
)

Best XGBoost parameters:
{'learning_rate': 0.15, 'max_depth': 6, 'n_estimators': 400}
Best mean cross-validation R²: 0.7185


In [22]:
# Convert all cross-validation results into a table
xgb_cv_results = pd.DataFrame(
    xgb_grid_search.cv_results_
)


xgb_cv_summary = (
    xgb_cv_results[
        [
            "param_max_depth",
            "param_learning_rate",
            "param_n_estimators",
            "mean_train_score",
            "std_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time"
        ]
    ]
    .rename(
        columns={
            "param_max_depth": "Max Depth",
            "param_learning_rate": "Learning Rate",
            "param_n_estimators": "N Estimators",
            "mean_train_score": "Mean Train R²",
            "std_train_score": "Train R² Std",
            "mean_test_score": "Mean CV R²",
            "std_test_score": "CV R² Std",
            "rank_test_score": "CV Rank",
            "mean_fit_time": "Mean Fit Time (Seconds)"
        }
    )
    .sort_values(
        [
            "CV Rank",
            "Mean CV R²"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


display(
    xgb_cv_summary.round(4)
)

,Max Depth,Learning Rate,N Estimators,Mean Train R²,Train R² Std,Mean CV R²,CV R² Std,CV Rank,Mean Fit Time (Seconds)
0,6,0.15,400,0.9649,0.0079,0.7185,0.0748,1,5.2012
1,6,0.15,300,0.9583,0.0089,0.7160,0.0742,2,4.0481
2,6,0.15,200,0.9476,0.0094,0.7109,0.0748,3,2.8402
3,7,0.15,400,0.9744,0.0064,0.7049,0.0776,4,6.3653
4,7,0.15,300,0.9696,0.0065,0.7030,0.0771,5,5.0059
5,6,0.10,400,0.9550,0.0085,0.7009,0.0641,6,5.0601
6,7,0.15,200,0.9613,0.0067,0.6993,0.0763,7,3.7749
7,6,0.10,300,0.9476,0.0092,0.6977,0.0638,8,4.1334
8,6,0.10,200,0.9361,0.0095,0.6917,0.0629,9,2.8741
9,6,0.20,400,0.9722,0.0069,0.6898,0.0613,10,4.9824


In [23]:
# Add a train-validation gap to inspect overfitting
xgb_cv_summary["Train-CV Gap"] = (
    xgb_cv_summary["Mean Train R²"]
    -
    xgb_cv_summary["Mean CV R²"]
)

display(
    xgb_cv_summary[
        [
            "Max Depth",
            "Learning Rate",
            "N Estimators",
            "Mean Train R²",
            "Mean CV R²",
            "Train-CV Gap",
            "CV R² Std",
            "CV Rank"
        ]
    ].round(4)
)

,Max Depth,Learning Rate,N Estimators,Mean Train R²,Mean CV R²,Train-CV Gap,CV R² Std,CV Rank
0,6,0.15,400,0.9649,0.7185,0.2463,0.0748,1
1,6,0.15,300,0.9583,0.7160,0.2423,0.0742,2
2,6,0.15,200,0.9476,0.7109,0.2367,0.0748,3
3,7,0.15,400,0.9744,0.7049,0.2695,0.0776,4
4,7,0.15,300,0.9696,0.7030,0.2666,0.0771,5
5,6,0.10,400,0.9550,0.7009,0.2541,0.0641,6
6,7,0.15,200,0.9613,0.6993,0.2620,0.0763,7
7,6,0.10,300,0.9476,0.6977,0.2499,0.0638,8
8,6,0.10,200,0.9361,0.6917,0.2444,0.0629,9
9,6,0.20,400,0.9722,0.6898,0.2824,0.0613,10


## Final XGBoost Evaluation

GridSearchCV automatically refits the best parameter combination using
the complete training set.

The final test set is then used once to estimate performance on the
held-out future month.

In [25]:
# Retrieve the best refitted model
best_xgb_model = xgb_grid_search.best_estimator_

best_xgb_model

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [26]:
# Generate final predictions
tuned_train_predictions = best_xgb_model.predict(
    X_train
)

tuned_test_predictions = best_xgb_model.predict(
    X_test
)


# Calculate final R² scores
tuned_train_r2 = r2_score(
    y_train,
    tuned_train_predictions
)

tuned_test_r2 = r2_score(
    y_test,
    tuned_test_predictions
)

tuned_train_test_gap = (
    tuned_train_r2
    -
    tuned_test_r2
)


print("Best parameters:")
print(xgb_grid_search.best_params_)

print(
    f"Best cross-validation R²: "
    f"{xgb_grid_search.best_score_:.4f}"
)

print(
    f"Tuned XGBoost training R²: "
    f"{tuned_train_r2:.4f}"
)

print(
    f"Tuned XGBoost testing R²: "
    f"{tuned_test_r2:.4f}"
)

print(
    f"Tuned train-test gap: "
    f"{tuned_train_test_gap:.4f}"
)

Best parameters:
{'learning_rate': 0.15, 'max_depth': 6, 'n_estimators': 400}
Best cross-validation R²: 0.7185
Tuned XGBoost training R²: 0.9476
Tuned XGBoost testing R²: 0.5454
Tuned train-test gap: 0.4023


Compare baseline and tuned model

In [27]:
xgb_model_comparison = pd.DataFrame(
    {
        "Model": [
            "Baseline XGBoost",
            "Tuned XGBoost"
        ],
        "Train R²": [
            baseline_train_r2,
            tuned_train_r2
        ],
        "Test R²": [
            baseline_test_r2,
            tuned_test_r2
        ],
        "Train-Test Gap": [
            baseline_gap,
            tuned_train_test_gap
        ]
    }
)


xgb_model_comparison["Test R² Change vs Baseline"] = (
    xgb_model_comparison["Test R²"]
    -
    baseline_test_r2
)


display(
    xgb_model_comparison.round(4)
)

,Model,Train R²,Test R²,Train-Test Gap,Test R² Change vs Baseline
0,Baseline XGBoost,0.8897,0.5192,0.3705,0.0000
1,Tuned XGBoost,0.9476,0.5454,0.4023,0.0262


Compare XGBoost With Previous Models

In [29]:
# Replace these values with the exact Week 6 updated-feature results
week6_linear_regression_train_r2 = 0.6616
week6_linear_regression_test_r2 = 0.4854

week6_decision_tree_train_r2 = 0.6958
week6_decision_tree_test_r2 = 0.4768

week6_random_forest_train_r2 = 0.8179
week6_random_forest_test_r2 = 0.5154

In [30]:
advanced_model_comparison = pd.DataFrame(
    {
        "Model": [
            "Linear Regression",
            "Decision Tree",
            "Random Forest",
            "Baseline XGBoost",
            "Tuned XGBoost"
        ],

        "Feature Set": [
            "Week 6 Updated Features",
            "Week 6 Updated Features",
            "Week 6 Updated Features",
            "Week 6 Updated Features",
            "Week 6 Updated Features"
        ],

        "Train R²": [
            week6_linear_regression_train_r2,
            week6_decision_tree_train_r2,
            week6_random_forest_train_r2,
            baseline_train_r2,
            tuned_train_r2
        ],

        "Test R²": [
            week6_linear_regression_test_r2,
            week6_decision_tree_test_r2,
            week6_random_forest_test_r2,
            baseline_test_r2,
            tuned_test_r2
        ]
    }
)


advanced_model_comparison["Train-Test Gap"] = (
    advanced_model_comparison["Train R²"]
    -
    advanced_model_comparison["Test R²"]
)


advanced_model_comparison = (
    advanced_model_comparison
    .sort_values(
        "Test R²",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    advanced_model_comparison.round(4)
)

,Model,Feature Set,Train R²,Test R²,Train-Test Gap
0,Tuned XGBoost,Week 6 Updated Features,0.9476,0.5454,0.4023
1,Baseline XGBoost,Week 6 Updated Features,0.8897,0.5192,0.3705
2,Random Forest,Week 6 Updated Features,0.8179,0.5154,0.3025
3,Linear Regression,Week 6 Updated Features,0.6616,0.4854,0.1762
4,Decision Tree,Week 6 Updated Features,0.6958,0.4768,0.2190
